# S14 — Attention

**Module 3**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s14_attention.ipynb)

Every cell below is a worked example from the [S14 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s14/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s14.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s14.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
print("environment ready")

## Two scoring functions, side by side


*Expected output starts with:* `dot-product: scores = [2.  0.  1.9]  weights = [0.4902 0.0663 0.4435]  output = [0.7119 `


In [ ]:
import numpy as np

np.random.seed(0)

def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

# Two scoring functions over the same keys and values. Dot-product
# (Luong-style) scoring is parameter-free; additive (Bahdanau-style)
# scoring runs query and key through a tiny one-hidden-layer MLP with
# its own weights W_q, W_k, v. Everything after the scores -- softmax,
# weighted average of values -- is identical.

d_k, d_att = 4, 8   # key dim; hidden size of the additive scorer

K = np.array([[1.0, 0.0, 1.0, 0.0],    # key 0: "animal-ish"
              [0.0, 1.0, 0.0, 1.0],    # key 1: "place-ish"
              [1.0, 0.0, 0.9, 0.1]])   # key 2: also "animal-ish"
V = np.array([[1.0, 0.0],
              [0.0, 1.0],
              [0.5, 0.5]])
q = np.array([2.0, 0.0, 2.0, 0.0])     # the "animal" query

def dot_scores(q, K):
    return K @ q / np.sqrt(d_k)

W_q = np.random.randn(d_att, d_k) * 0.5
W_k = np.random.randn(d_att, d_k) * 0.5
v_a = np.random.randn(d_att) * 0.5

def additive_scores(q, K):
    # score(q, k_i) = v^T tanh(W_q @ q + W_k @ k_i)
    return np.tanh(W_q @ q + K @ W_k.T) @ v_a

for name, scores in [("dot-product", dot_scores(q, K)),
                     ("additive   ", additive_scores(q, K))]:
    w = softmax(scores)
    out = w @ V
    print(f"{name}: scores = {np.round(scores, 4)}  weights = {np.round(w, 4)}"
          f"  output = {np.round(out, 4)}")

n_params = W_q.size + W_k.size + v_a.size
print(f"\nadditive scorer parameters: {n_params} (dot-product scorer: 0)")

# Cost of scoring T queries against T keys. Dot-product: one length-d_k
# dot per pair, a single matmul. Additive (with W_q @ q and W_k @ k
# precomputed for all positions): ~2*d_att ops per pair, plus a
# (T, T, d_att) intermediate if vectorized.
for T in [512, 4096]:
    dot = T * T * d_k
    add = T * T * 2 * d_att + 2 * T * d_att * d_k
    print(f"T = {T:>4}: dot-product ~{dot/1e6:>7.1f}M ops, "
          f"additive ~{add/1e6:>7.1f}M ops, "
          f"additive intermediate: {T*T*d_att*4/1e9:.2f} GB float32")

## Scaled dot-product attention in NumPy


*Expected output starts with:* `query ~ animal: weights = [0.4902 0.0663 0.4435]  ->  output = [0.7119 0.2881]`


In [ ]:
import numpy as np

np.random.seed(0)

def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)  # numerical stability
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    d_k = K.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)      # (n_q, n_k)
    weights = softmax(scores, axis=-1)   # each row sums to 1
    return weights @ V, weights

# A hand-built example with d_k = 4. Keys 0 and 2 point in similar
# directions; key 1 points somewhere else entirely.
K = np.array([[1.0, 0.0, 1.0, 0.0],    # key 0: "animal-ish"
              [0.0, 1.0, 0.0, 1.0],    # key 1: "place-ish"
              [1.0, 0.0, 0.9, 0.1]])   # key 2: also "animal-ish"
V = np.array([[1.0, 0.0],              # value carried by key 0
              [0.0, 1.0],              # value carried by key 1
              [0.5, 0.5]])             # value carried by key 2

queries = {
    "query ~ animal": np.array([[2.0, 0.0, 2.0, 0.0]]),
    "query ~ place":  np.array([[0.0, 2.0, 0.0, 2.0]]),
    "query ~ mixed":  np.array([[1.0, 1.0, 1.0, 1.0]]),
}

for name, Q in queries.items():
    out, w = scaled_dot_product_attention(Q, K, V)
    print(f"{name}: weights = {np.round(w[0], 4)}  ->  output = {np.round(out[0], 4)}")

## Why divide by sqrt(d_k)


*Expected output starts with:* `  d_k   std of q.k   max weight (raw)   max weight (scaled)`


In [ ]:
import numpy as np

np.random.seed(0)

def softmax(z):
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

# Dot products of random unit-variance vectors grow with dimension:
# if q, k have i.i.d. N(0,1) entries, q . k has variance d_k.
# Without scaling, softmax over 16 keys saturates as d_k grows.
n_keys, trials = 16, 200

print(f"{'d_k':>5}  {'std of q.k':>11}  {'max weight (raw)':>17}  {'max weight (scaled)':>20}")
for d_k in [4, 16, 64, 256, 1024]:
    max_raw, max_scaled, dots_all = [], [], []
    for _ in range(trials):
        q = np.random.randn(d_k)
        K = np.random.randn(n_keys, d_k)
        dots = K @ q
        dots_all.append(dots)
        max_raw.append(softmax(dots).max())
        max_scaled.append(softmax(dots / np.sqrt(d_k)).max())
    print(f"{d_k:>5}  {np.concatenate(dots_all).std():>11.2f}  "
          f"{np.mean(max_raw):>17.4f}  {np.mean(max_scaled):>20.4f}")

# Saturated softmax means dead gradients. The Jacobian of softmax has
# entries involving p_i * (1 - p_i); when one p_i ~ 1, all entries ~ 0.
print("\ngradient factor p*(1-p) at the winning key:")
for d_k in [4, 256]:
    np.random.seed(0)
    q = np.random.randn(d_k)
    K = np.random.randn(n_keys, d_k)
    for label, logits in [("raw", K @ q), ("scaled", K @ q / np.sqrt(d_k))]:
        p = softmax(logits)
        i = p.argmax()
        print(f"  d_k = {d_k:>4}, {label:>6}: max p = {p[i]:.6f}, p*(1-p) = {p[i]*(1-p[i]):.6f}")

## The price of looking everywhere


*Expected output starts with:* `    T  score+mix MACs  vs prev  score matrix`


In [ ]:
import time
import numpy as np

np.random.seed(0)

# Self-attention cost versus sequence length. For T tokens the score
# matrix Q @ K^T has T^2 entries, so both the arithmetic and the memory
# of the attention step grow quadratically -- the table counts them
# exactly. The measurement at the end times the real computation at
# doubling lengths and classifies the observed growth; absolute times
# depend on the machine, so only the growth rate per doubling is
# reported.

def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V):
    w = softmax(Q @ K.T / np.sqrt(Q.shape[-1]), axis=-1)
    return w @ V

d = 64
lengths = [256, 512, 1024, 2048, 4096]

print(f"{'T':>5}  {'score+mix MACs':>14}  {'vs prev':>7}  {'score matrix':>12}")
prev = None
for T in lengths:
    macs = 2 * T * T * d            # Q@K^T plus weights@V: d MACs per entry each
    mem = T * T * 4 / 1e6           # float32 score matrix, in MB
    ratio = f"{macs / prev:.1f}x" if prev else "-"
    print(f"{T:>5}  {macs / 1e6:>13.0f}M  {ratio:>7}  {mem:>9.1f} MB")
    prev = macs

# Now measure wall-clock time (best of 5 runs at each length).
best = []
for T in lengths:
    Q = np.random.randn(T, d).astype(np.float32)
    K = np.random.randn(T, d).astype(np.float32)
    V = np.random.randn(T, d).astype(np.float32)
    attention(Q, K, V)              # warm-up
    reps = []
    for _ in range(5):
        t0 = time.perf_counter()
        attention(Q, K, V)
        reps.append(time.perf_counter() - t0)
    best.append(min(reps))

# Consecutive timing ratios bounce with machine noise, so take the
# median growth factor per doubling and bucket it against the candidates:
# linear -> ~2x, quadratic -> ~4x, cubic -> ~8x.
ratios = sorted(b / a for a, b in zip(best, best[1:]))
med = ratios[len(ratios) // 2]
if med < 3.0:
    verdict = "linear-like (about 2x per doubling)"
elif med < 6.0:
    verdict = "quadratic-like (about 4x per doubling)"
else:
    verdict = "cubic-or-worse (8x or more per doubling)"
print(f"\nmeasured median time growth per doubling of T: {verdict}")

## Attention at generation time: the KV cache


*Expected output starts with:* `max |naive - cached| over all 16 steps: 6.66e-16`


In [ ]:
import numpy as np

np.random.seed(0)

# The KV cache. During generation, a causal model produces one token at a
# time, and the new token's attention needs the keys and values of every
# earlier token. Recomputing them from scratch each step repeats work the
# model already did; caching them makes each step's projection cost
# constant. This script verifies that the cached computation produces the
# same numbers, then counts the work saved.

d, T = 32, 16
W_q, W_k, W_v = (np.random.randn(d, d) / np.sqrt(d) for _ in range(3))
X = np.random.randn(T, d)          # the token representations, step by step

def softmax(z):
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

def attend(q, K, V):
    return softmax(K @ q / np.sqrt(d)) @ V

# Version 1: no cache. At each step t, re-project ALL tokens so far.
outputs_naive, proj_macs_naive = [], 0
for t in range(1, T + 1):
    K = X[:t] @ W_k                # recomputed from scratch every step
    V = X[:t] @ W_v
    q = X[t - 1] @ W_q
    proj_macs_naive += (2 * t + 1) * d * d
    outputs_naive.append(attend(q, K, V))

# Version 2: KV cache. Project only the NEW token; append to the cache.
K_cache = np.zeros((0, d)); V_cache = np.zeros((0, d))
outputs_cached, proj_macs_cached = [], 0
for t in range(1, T + 1):
    k_new = X[t - 1] @ W_k         # one row, not t rows
    v_new = X[t - 1] @ W_v
    q = X[t - 1] @ W_q
    proj_macs_cached += 3 * d * d
    K_cache = np.vstack([K_cache, k_new])
    V_cache = np.vstack([V_cache, v_new])
    outputs_cached.append(attend(q, K_cache, V_cache))

diff = max(np.abs(a - b).max() for a, b in zip(outputs_naive, outputs_cached))
print(f"max |naive - cached| over all {T} steps: {diff:.2e}")
print(f"projection MACs, naive:  {proj_macs_naive:>9,}")
print(f"projection MACs, cached: {proj_macs_cached:>9,}")
print(f"savings factor at T = {T}: {proj_macs_naive / proj_macs_cached:.1f}x "
      f"(grows with T: the naive version repeats old work every step)")
print(f"cache size at T = {T}: 2 * {T} * {d} floats per attention layer")

## Try it yourself

1. In the NumPy example, scale the animal query from `[2, 0, 2, 0]` up to `[8, 0, 8, 0]` and rerun. What happens to the weight distribution, and what does this tell you about how a trained model can sharpen its own attention?
2. Add a fourth key/value pair that duplicates key 0 exactly. How does the animal query's weight redistribute, and why does the output change even though you added no new "information"?
3. Reproduce the saturation table with 4 keys and with 256 keys instead of 16. Does the number of keys change how badly unscaled attention saturates, or only the baseline `1/n_keys` weight?
4. The additive-scorer example used random weights. Train them: with the animal/place keys and values fixed, define a target output for each query and fit `W_q`, `W_k`, `v_a` by gradient descent (autograd via PyTorch, or finite differences). How closely can the trained additive scorer reproduce the dot-product scorer's weights?
5. In the complexity script, hold `T = 1024` fixed and double `d` from 16 to 256. Predict the growth rate of the MAC count before running, then check whether measured time follows — and explain why the score-matrix memory column does not move at all.
6. Extend the KV-cache script to also count the *score and mix* MACs per step. Plot (or print) per-step totals with and without the cache: which part of generation does the cache make constant, and which part still grows with `t`?


---

Full discussion of everything above: [S14 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s14/).
